# Hotel Booking Capstone Project

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

## Step 2: Load Dataset

In [ ]:
df = pd.read_csv("hotel_bookings(6).csv")
df.head()

## Step 3: Dataset Shape

In [ ]:
df.shape

## Step 4: Dataset Information

In [ ]:
df.info()

## Step 5: Statistical Summary

In [ ]:
df.describe(include="all").T

## Step 6: Missing Values

In [ ]:
df.isnull().sum().sort_values(ascending=False)

## Step 7: Remove Duplicates

In [ ]:
df = df.drop_duplicates().copy()
df.shape

## Step 8: Handle Missing Values

In [ ]:
df["children"] = df["children"].fillna(0)
df["country"] = df["country"].fillna("Unknown")
df["agent"] = df["agent"].fillna(0)
df["company"] = df["company"].fillna(0)

df.isnull().sum().sort_values(ascending=False).head()

## Step 9: Check Invalid Values

In [ ]:
df[["adults", "children", "babies", "adr",
    "stays_in_weekend_nights", "stays_in_week_nights"]].describe()

## Step 10: Remove Invalid Records

In [ ]:
df = df[
    (df["adults"] >= 0) &
    (df["children"] >= 0) &
    (df["babies"] >= 0) &
    (df["adr"] >= 0) &
    (df["stays_in_weekend_nights"] >= 0) &
    (df["stays_in_week_nights"] >= 0)
].copy()

df.shape

## Step 11: Exploratory Data Analysis

In [ ]:
df["hotel"].value_counts().plot(kind="bar")
plt.title("Bookings by Hotel Type")
plt.xlabel("Hotel")
plt.ylabel("Bookings")
plt.show()

## Step 12: Cancellation Analysis

In [ ]:
sns.countplot(data=df, x="is_canceled")
plt.title("Booking Cancellation")
plt.show()

## Step 13: Monthly Bookings

In [ ]:
month_order = ["January", "February", "March", "April", "May", "June",
               "July", "August", "September", "October", "November", "December"]

df["arrival_date_month"].value_counts().reindex(month_order).plot(kind="bar", figsize=(10,4))
plt.title("Bookings by Month")
plt.xticks(rotation=45)
plt.show()

## Step 14: Booking Price Analysis

In [ ]:
sns.histplot(df["adr"], bins=50)
plt.title("Average Daily Rate")
plt.show()

## Step 15: Feature Engineering

In [ ]:
df["total_guests"] = df["adults"] + df["children"] + df["babies"]
df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["is_family"] = ((df["children"] + df["babies"]) > 0).astype(int)
df["has_special_request"] = (df["total_of_special_requests"] > 0).astype(int)

df[["total_guests", "total_nights", "is_family", "has_special_request"]].head()

## Step 16: Select Features

In [ ]:
numeric_features = [
    "lead_time", "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "children", "babies", "booking_changes",
    "previous_cancellations", "previous_bookings_not_canceled",
    "days_in_waiting_list", "adr", "required_car_parking_spaces",
    "total_of_special_requests", "total_guests", "total_nights",
    "is_family", "has_special_request", "is_repeated_guest"
]

categorical_features = [
    "hotel", "meal", "market_segment", "distribution_channel",
    "reserved_room_type", "deposit_type", "customer_type"
]

X_raw = df[numeric_features + categorical_features].copy()

## Step 17: Preprocessing

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X = preprocessor.fit_transform(X_raw)
X.shape

## Step 18: K-Means — Find Best K

In [ ]:
sample_size = min(10000, len(X))
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(X), sample_size, replace=False)
X_sample = X[sample_idx]

inertia = []
silhouette = []

for k in range(2, 9):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_sample)
    inertia.append(model.inertia_)
    silhouette.append(silhouette_score(X_sample, labels))

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(range(2,9), inertia, marker="o")
plt.title("Elbow Method")
plt.xlabel("K")
plt.ylabel("Inertia")

plt.subplot(1,2,2)
plt.plot(range(2,9), silhouette, marker="o")
plt.title("Silhouette Score")
plt.xlabel("K")
plt.ylabel("Score")
plt.show()

## Step 19: Train K-Means

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df["kmeans_cluster"] = kmeans.fit_predict(X)

df["kmeans_cluster"].value_counts().sort_index()

## Step 20: K-Means Cluster Analysis

In [ ]:
df.groupby("kmeans_cluster")[[
    "lead_time", "adr", "total_guests", "total_nights",
    "total_of_special_requests", "is_repeated_guest"
]].mean().round(2)

## Step 21: Hierarchical Clustering

In [ ]:
hier_size = min(2000, len(X_sample))
hier_idx = rng.choice(len(X_sample), hier_size, replace=False)
X_hier = X_sample[hier_idx]

hierarchical = AgglomerativeClustering(n_clusters=4)
hier_labels = hierarchical.fit_predict(X_hier)

pd.Series(hier_labels).value_counts().sort_index()

## Step 22: DBSCAN

In [ ]:
db_size = min(5000, len(X_sample))
db_idx = rng.choice(len(X_sample), db_size, replace=False)
X_db = X_sample[db_idx]

dbscan = DBSCAN(eps=3.0, min_samples=10)
db_labels = dbscan.fit_predict(X_db)

pd.Series(db_labels).value_counts().sort_index()

## Step 23: Gaussian Mixture Model

In [ ]:
gmm_size = min(10000, len(X_sample))
gmm_idx = rng.choice(len(X_sample), gmm_size, replace=False)
X_gmm = X_sample[gmm_idx]

gmm = GaussianMixture(n_components=4, random_state=42)
gmm_labels = gmm.fit_predict(X_gmm)

pd.Series(gmm_labels).value_counts().sort_index()

## Step 24: PCA

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

print("Explained variance:", pca.explained_variance_ratio_)

In [ ]:
plot_size = min(10000, len(X_pca))
plot_idx = rng.choice(len(X_pca), plot_size, replace=False)

plt.figure(figsize=(8,6))
sns.scatterplot(
    x=X_pca[plot_idx, 0],
    y=X_pca[plot_idx, 1],
    hue=df["kmeans_cluster"].to_numpy()[plot_idx],
    palette="viridis",
    s=15
)
plt.title("K-Means Clusters using PCA")
plt.show()

## Step 25: Isolation Forest

In [ ]:
iso_size = min(20000, len(X))
iso_idx = rng.choice(len(X), iso_size, replace=False)

isolation_forest = IsolationForest(
    contamination=0.02,
    random_state=42,
    n_estimators=150
)

anomaly_labels = isolation_forest.fit_predict(X[iso_idx])
df.loc[df.index[iso_idx], "anomaly"] = np.where(anomaly_labels == -1, 1, 0)

df["anomaly"].value_counts(dropna=False)

## Step 26: Cancellation Rate by Cluster

In [ ]:
df.groupby("kmeans_cluster")["is_canceled"].mean().mul(100).round(2)

## Step 27: Hotel Type by Cluster

In [ ]:
pd.crosstab(df["kmeans_cluster"], df["hotel"], normalize="index").round(2)

## Step 28: Market Segment by Cluster

In [ ]:
pd.crosstab(df["kmeans_cluster"], df["market_segment"], normalize="index").round(2)

## Step 29: Save Results

In [ ]:
df.to_csv("hotel_booking_capstone_results.csv", index=False)
print("Results saved successfully.")

## Step 30: Final Summary

In [ ]:
print("Final dataset shape:", df.shape)
print("K-Means clusters:", df["kmeans_cluster"].nunique())
print("Rows flagged as anomalies:", int((df["anomaly"] == 1).sum()))
print("PCA explained variance:", pca.explained_variance_ratio_.round(3))